# DynLang-SLAM: Language Pipeline on Colab
Make sure you're using a **GPU runtime** (Runtime → Change runtime type → A100/T4)

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f'torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
# Install dependencies
!pip install gsplat sam2 open_clip_torch omegaconf -q

In [ ]:
# Verify gsplat
from gsplat import rasterization
print('gsplat OK')

# Pre-download CLIP ViT-L/14 from OpenAI CDN (no HuggingFace auth needed)
import os, urllib.request
clip_cache = os.path.expanduser("~/.cache/clip")
os.makedirs(clip_cache, exist_ok=True)
clip_path = os.path.join(clip_cache, "ViT-L-14.pt")
if not os.path.exists(clip_path):
    print("Downloading CLIP ViT-L/14 from OpenAI CDN (~890MB)...")
    urllib.request.urlretrieve(
        "https://openaipublic.azureedge.net/clip/models/"
        "b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836/ViT-L-14.pt",
        clip_path
    )
    print(f"CLIP weights saved to {clip_path}")
else:
    print(f"CLIP weights already cached at {clip_path}")
print(f"  Size: {os.path.getsize(clip_path)/1e6:.0f} MB")

In [ ]:
# Clone project from GitHub
!git clone https://github.com/ankur-cap-kirk/DynLang-SLAM.git /content/DynLang-SLAM 2>/dev/null || echo "Already cloned"
!ls /content/DynLang-SLAM/

In [ ]:
# Download SAM2 checkpoint
!mkdir -p /content/DynLang-SLAM/checkpoints
!wget -q -O /content/DynLang-SLAM/checkpoints/sam2.1_hiera_tiny.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt
!ls -lh /content/DynLang-SLAM/checkpoints/

In [ ]:
# Set multi-scale in config (Colab has enough VRAM)
import yaml
with open('/content/DynLang-SLAM/configs/default.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
cfg['language']['scales'] = ['whole', 'part', 'subpart']
with open('/content/DynLang-SLAM/configs/default.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
print('Config updated: multi-scale enabled')

In [ ]:
# Download Replica room0 dataset
# Option A: From Google Drive (if you uploaded Replica-room0.zip)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

!mkdir -p /content/DynLang-SLAM/data/Replica
!cp /content/drive/MyDrive/Replica-room0.zip /content/ 2>/dev/null && \
  cd /content && unzip -qo Replica-room0.zip -d /content/DynLang-SLAM/data/Replica/ || \
  echo "No Replica-room0.zip on Drive. Download with Option B below."

# Option B: Download directly (uncomment if no Drive zip)
# !mkdir -p /content/DynLang-SLAM/data/Replica && cd /content/DynLang-SLAM/data/Replica && \
#   wget -q https://cvg-data.inf.ethz.ch/nice-slam/data/Replica/room0.zip && unzip -qo room0.zip

!ls /content/DynLang-SLAM/data/Replica/room0/results/ | head -3

In [ ]:
# Run standalone language test
%cd /content/DynLang-SLAM
!python scripts/test_language.py

In [ ]:
# Run full language SLAM test
!python scripts/test_language_slam.py

In [ ]:
# View results
from IPython.display import Image, display
import glob
for img in sorted(glob.glob('results/language_test/query_*_relevancy.jpg')):
    print(img.split('/')[-1])
    display(Image(img, width=600))